## 1. Data Ingestion

Crime data was downloaded from the UK Police Data website for four police forces: Dorset, Wiltshire, Gloucestershire and Warwickshire, covering January 2024 to December 2025.
The data consists of 96 monthly CSV files, organised into a single folder. Python's `pathlib` library is used to locate the files automatically without specifying individual file paths.

The files are read in batches of 10,000 records and temporarily stored in a SQLite database. This avoids holding all 96 source DataFrames in memory simultaneously.The staged records are then loaded into Pandas to allow cross file cleaning, duplicate detection and geographical validation. The temporary database is deleted after processing.

Validation confirmed that all 437,271 records were loaded, with the original missing values and duplicate counts preserved.

In [1]:
from pathlib import Path
import pandas as pd

data_folder = Path("Crime data")
crime_files = sorted(data_folder.glob("*.csv"))

print(f"Found {len(crime_files)} crime files")
assert len(crime_files) == 96, "Unexpected number of crime files"

Found 96 crime files


In [ ]:
sample = pd.read_csv(crime_files[0])

print(sample.shape)
print(sample.columns.tolist())
sample.head()

### Initial Source Data Inspection

The January 2024 Dorset file was selected as an initial sample to inspect the dataset structure, column types, missing values and crime categories before processing all 96 files.
This initial inspection helps identify potential data quality issues requiring further investigation across the full dataset.

In [ ]:
sample.info()

In [9]:
sample.isna().sum()

NameError: name 'sample' is not defined

In [ ]:
sample["Crime type"].value_counts()

In [ ]:
sample.groupby("Crime type")["Crime ID"].apply(lambda x: x.isna().sum())

In [5]:
import sqlite3
from tempfile import TemporaryDirectory

total_loaded = 0

with TemporaryDirectory() as temp_dir:
    db_path = Path(temp_dir) / "crime_staging.db"

    conn = sqlite3.connect(db_path)

    try:
        for file in crime_files:
            file_rows = 0

            for batch in pd.read_csv(file, chunksize=10000):
                batch.to_sql("crime_records",conn,if_exists="append",index=False)
                file_rows += len(batch)

            total_loaded += file_rows

        # Validate number of records stored
        staged_rows = conn.execute("SELECT COUNT(*) FROM crime_records").fetchone()[0]

        print("Staged records:", staged_rows)
        print("Loaded records:", total_loaded)
        assert staged_rows == total_loaded

        #load combined data for cross file validation
        crime_df = pd.read_sql_query("SELECT * FROM crime_records ORDER BY rowid",conn)

    finally:
        conn.close()

print(f"Files processed: {len(crime_files)}")
print(f"Records loaded: {total_loaded:,}")
print(f"Columns: {len(crime_df.columns)}")

assert len(crime_df) == total_loaded

Staged records: 437271
Loaded records: 437271
Files processed: 96
Records loaded: 437,271
Columns: 12


In [7]:
print(crime_df["Reported by"].value_counts())
print(crime_df["Month"].nunique())

Reported by
Gloucestershire Constabulary    117553
Dorset Police                   117392
Wiltshire Police                102071
Warwickshire Police             100255
Name: count, dtype: int64
24


In [8]:
crime_df.groupby("Crime type")["Crime ID"].apply(lambda x: x.isna().sum())

Crime type
Anti-social behaviour           91587
Bicycle theft                       0
Burglary                            0
Criminal damage and arson           0
Drugs                               0
Other crime                         0
Other theft                         0
Possession of weapons               0
Public order                        0
Robbery                             0
Shoplifting                         0
Theft from the person               0
Vehicle crime                       0
Violence and sexual offences        0
Name: Crime ID, dtype: int64

### Data Quality Findings

An initial inspection of the January 2024 Dorset dataset identified 958 missing Crime IDs and corresponding missing outcome categories. All 958 records were classified as antisocial behaviour.

This finding was then checked across all 96 source files containing 437,271 records.  91,587 records had missing Crime IDs, all classified as antisocial behaviour. No other crime categories had missing Crime IDs.

These missing values appear to reflect differences in how anti social behaviour incidents are recorded rather than data corruption. These records will be retained for reporting with anti social behaviour distinguished from recorded crimes.

In [ ]:
print("Duplicate rows:", crime_df.duplicated().sum())

print("Duplicate Crime IDs:", crime_df.loc[crime_df["Crime ID"].notna(), "Crime ID"].duplicated().sum())

In [ ]:
duplicates = crime_df[crime_df.duplicated()]

duplicates["Crime type"].value_counts()

In [ ]:
crime_df[crime_df["Crime ID"].notna()].duplicated(subset=["Crime ID", "Month"]).sum()

In [ ]:
repeated_crimes = crime_df[crime_df["Crime ID"].notna()& crime_df.duplicated(
    subset=["Crime ID", "Month"],
    keep=False
    )]

repeated_crimes.sort_values(["Crime ID", "Month"]).head(10)

In [ ]:
crime_df[crime_df["Crime ID"].notna()].groupby(["Crime ID", "Month"])["Crime type"].nunique().value_counts()

In [ ]:
category_counts = crime_df[crime_df["Crime ID"].notna()].groupby(["Crime ID", "Month"])["Crime type"].nunique()

conflicting_ids = category_counts[category_counts > 1].index

crime_df.set_index(["Crime ID", "Month"]).loc[conflicting_ids, ["Reported by", "Crime type", "Last outcome category"]]

In [ ]:
crime_df[crime_df["Crime ID"].notna()].groupby(["Crime ID", "Month"])["Reported by"].nunique().value_counts()

## 3. Data Cleaning and Transformation

In [ ]:
# Separate ASB incidents from recorded crimes
asb = crime_df[crime_df["Crime type"] == "Anti-social behaviour"].copy()

crimes = crime_df[crime_df["Crime type"] != "Anti-social behaviour"].copy()

# Remove the five ambiguous Crime IDs
crimes = crimes[~crimes.set_index(["Crime ID", "Month"]).index.isin(conflicting_ids)].copy()

# Retain one record per identified crime, force and month
crimes = crimes.drop_duplicates(subset=["Crime ID", "Month", "Reported by"])

# Recombine the cleaned records
cleaned_df = pd.concat([crimes, asb], ignore_index=True)

print(f"Original records: {len(crime_df):,}")
print(f"Cleaned records: {len(cleaned_df):,}")
print(f"Records removed: {len(crime_df) - len(cleaned_df):,}")

In [ ]:
# Check for remaining duplicates in recorded crimes
remaining_duplicates = cleaned_df[cleaned_df["Crime ID"].notna()].duplicated(subset=["Crime ID", "Month", "Reported by"]).sum()

print("Remaining duplicate crimes:", remaining_duplicates)

assert remaining_duplicates == 0
assert len(crime_df) == len(cleaned_df) + 579

### Cleaning Decisions

Duplicate records were investigated before removal. Most identical rows related to antisocial behaviour which lacks unique Crime IDs. These records were retained because separate incidents cannot reliably be distinguished.

For recorded crimes repeated Crime IDs within the same force and month were treated as duplicate representations of the same crime. Five Crime IDs had conflicting crime categories and were excluded because their correct classification could not be established.

Cleaning reduced the dataset from 437,271 to 436,692 records. The original data was retained separately to allow comparison and validation.

## 4. Aggregation for Reporting

In [ ]:
reporting_df = (cleaned_df.groupby(["Reported by", "Month", "Crime type"]).size().reset_index(name="Crime count"))

reporting_df.head(10)

In [ ]:
# Each cleaned record should contribute to one group
assert reporting_df["Crime count"].sum() == len(cleaned_df)

# Each force/month/category combination must be unique
assert not reporting_df.duplicated(subset=["Reported by", "Month", "Crime type"]).any()

# Check the dataset
print("Reporting rows:", len(reporting_df))
print("Total inncidents:", reporting_df["Crime count"].sum())
print("Police forces:", reporting_df["Reported by"].nunique())
print("Months:", reporting_df["Month"].nunique())

In [ ]:
reporting_df.head(15)

### Aggregation Results

The cleaned data was aggregated by police force month and crime category producing 1,344 reporting rows from 436,692 records.Each row represents one crime category within a specific police force and month. Validation confirmed that all cleaned records were accounted for and that no duplicate combinations existed at the chosen reporting grain.

Anti social behaviour incidents remain included but will be distinguished from recorded crimes in the final dataset.

## 5. Population Data Enrichment

In [ ]:
population_file = "policeforceareasmid2011tomid2025.xlsx"

excel_file = pd.ExcelFile(population_file)

print(excel_file.sheet_names)

In [ ]:
population_sample = pd.read_excel(population_file, sheet_name="Mid-2011 to Mid-2025", header=None, nrows=15)

population_sample

In [ ]:
population_df = pd.read_excel(population_file, sheet_name="Mid-2011 to Mid-2025", header=3)

population_df.head()

In [ ]:
print(population_df.columns.tolist())

In [ ]:
# Identify all age and sex columns
age_columns = [col for col in population_df.columns if col.startswith(("F", "M")) and col[1:].isdigit()]

# Sum population across all ages and both sexes
population_df["Population"] = population_df[age_columns].sum(axis=1)

population_df[["PFA 2023 Name", "Mid-Year", "Population"]].head()

In [ ]:
population_df["PFA 2023 Name"].unique()

In [ ]:
forces = ["Dorset", "Wiltshire", "Gloucestershire", "Warwickshire"]

population_clean = population_df[ population_df["PFA 2023 Name"].isin(forces) & population_df["Mid-Year"].isin(["mid-2024", "mid-2025"])][["PFA 2023 Name", "Mid-Year", "Population"]].copy()

population_clean

In [ ]:
print("Population rows:", len(population_clean))

print("Missing populations:", population_clean["Population"].isna().sum())

print( "Duplicate force/year combinations:", population_clean.duplicated(subset=["PFA 2023 Name", "Mid-Year"]).sum())

assert len(population_clean) == 8
assert population_clean["Population"].gt(0).all()

In [ ]:
# Match ONS force names to crime dataset
force_mapping = {
    "Dorset": "Dorset Police",
    "Wiltshire": "Wiltshire Police",
    "Gloucestershire": "Gloucestershire Constabulary",
    "Warwickshire": "Warwickshire Police"}

population_clean["Reported by"] = (population_clean["PFA 2023 Name"].map(force_mapping))

# Extract the year from values like"mid-2024"
population_clean["Year"] = (population_clean["Mid-Year"].str[-4:].astype(int))

# Extract the year from the monthly crime data
reporting_df["Year"] = (reporting_df["Month"].str[:4].astype(int))
population_clean[["Reported by", "Year", "Population"]]

In [ ]:
#join datasets 
enriched_df = reporting_df.merge(
    population_clean[["Reported by", "Year", "Population"]],
    on=["Reported by", "Year"],
    how="left",
    validate="many_to_one",
    indicator=True)

print(enriched_df["_merge"].value_counts())
print("Rows before:", len(reporting_df))
print("Rows after:", len(enriched_df))

In [ ]:
# Validate the join
assert (enriched_df["_merge"] == "both").all()
assert len(enriched_df) == len(reporting_df)
assert enriched_df["Population"].notna().all()
assert enriched_df["Population"].gt(0).all()

enriched_df = enriched_df.drop(columns="_merge")

In [ ]:
# rename count to reflect inclusion of ASB incidents
enriched_df = enriched_df.rename(columns={"Crime count": "Incident count"})

# Distinguish recorded crimes from ASB incidents
enriched_df["Incident type"] = enriched_df["Crime type"].apply(lambda x: "ASB" if x == "Anti-social behaviour" else "Recorded crime")

#Calculate category specific monthly rates
enriched_df["Rate per 1000"] = (enriched_df["Incident count"] / enriched_df["Population"] * 1000).round(2)

enriched_df.head()

In [ ]:
#Validate calculated rates
assert enriched_df["Rate per 1000"].notna().all()
assert enriched_df["Rate per 1000"].ge(0).all()

assert enriched_df["Incident count"].sum() == len(cleaned_df)

assert not enriched_df.duplicated(subset=["Reported by", "Month", "Crime type"]).any()

print("Reporting rows:", len(enriched_df))
print("Total incidents:", enriched_df["Incident count"].sum())

### Population Enrichment

Population estimates for 2024 and 2025 were obtained from the Office for National Statistics. Age and sex population figures were summed to calculate total population for each police force area.

Police force names were standardised to match the crime dataset. Population figures were joined using police force and year with validation confirming that all 1,344 reporting rows matched successfully without introducing duplicates.

Monthly incident rates per 1000 residents were calculated to support comparisons between police forces of different population sizes. Annual mid year population estimates were used for all months within the corresponding year assuming that population changes within each year are relatively small.

Antisocial behaviour incidents are identified separately from recorded crimes to avoid misleading comparisons.

## 6. Socioeconomic Enrichment

In [ ]:
deprivation_file = "File_7_IoD2025_All_Ranks_Scores_Deciles_Population_Denominators.csv"

deprivation_df = pd.read_csv(deprivation_file)

print(deprivation_df.shape)

deprivation_df[["LSOA code (2021)","Local Authority District name (2024)", "Income Score (rate)", "Total population: mid 2022"]].head()

In [ ]:
deprivation_df[deprivation_df["Local Authority District name (2024)"].str.contains("Dorset|Bournemouth|Poole|Wiltshire|Gloucester|Warwick",case=False,na=False)]["Local Authority District name (2024)"].unique()

In [ ]:
lookup_df = pd.read_excel("lookuptablefortheukauthoritycodesfor2024.xlsx")

lookup_df[["LAD24CD", "LAD24NM", "PFA24NM"]].head()

In [ ]:
selected_forces = ["Dorset","Wiltshire","Gloucestershire","Warwickshire"]

force_lookup = lookup_df[lookup_df["PFA24NM"].isin(selected_forces)][["LAD24CD", "LAD24NM", "PFA24NM"]].drop_duplicates()

print(force_lookup.groupby("PFA24NM").size())
print("Duplicate local authority codes:", force_lookup["LAD24CD"].duplicated().sum())

force_lookup

In [ ]:
[col for col in deprivation_df.columns if "Local Authority" in col]

In [ ]:
#Join the deprivation data to the lookup

deprivation_joined = deprivation_df.merge(
    force_lookup,
    left_on="Local Authority District code (2024)",
    right_on="LAD24CD",
    how="inner",
    validate="many_to_one")

print("Matched LSOAs:", len(deprivation_joined))
print(deprivation_joined["PFA24NM"].value_counts())

In [ ]:
# Validate the join

print("Missing income scores:", deprivation_joined["Income Score (rate)"].isna().sum())
print("Missing population denominators:", deprivation_joined["Total population: mid 2022"].isna().sum())
print("Local authorities matched:", deprivation_joined["LAD24CD"].nunique())
print("Police forces matched:", deprivation_joined["PFA24NM"].nunique())

In [ ]:
# calculate population weighted deprivation

# Ensure population denominators are valid
assert deprivation_joined["Total population: mid 2022"].gt(0).all()

#each LSOA should appear once
assert not deprivation_joined["LSOA code (2021)"].duplicated().any()

# Confirm all 15 local authorities are represented
assert set(force_lookup["LAD24CD"]) == set(deprivation_joined["LAD24CD"])

# Calculate populationweighted contributions
deprivation_joined["Weighted income"] = (deprivation_joined["Income Score (rate)"] * deprivation_joined["Total population: mid 2022"])

# Aggregate to police force level
force_deprivation = deprivation_joined.groupby("PFA24NM", as_index=False).agg(
    weighted_income=("Weighted income", "sum"),
    total_population=("Total population: mid 2022", "sum"))

force_deprivation["Income deprivation rate"] = (force_deprivation["weighted_income"] / force_deprivation["total_population"])

force_deprivation = force_deprivation[["PFA24NM", "Income deprivation rate"]]

force_deprivation

In [ ]:
# validate result 

assert len(force_deprivation) == 4
assert force_deprivation["Income deprivation rate"].between(0, 1).all()
assert force_deprivation["PFA24NM"].is_unique

In [ ]:
# Join deprivation data to reporting dataset

force_deprivation["Reported by"] = (force_deprivation["PFA24NM"].map(force_mapping))

final_df = enriched_df.merge(
    force_deprivation[["Reported by", "Income deprivation rate"]
    ],
    on="Reported by",
    how="left",
    validate="many_to_one",
    indicator=True)

print(final_df["_merge"].value_counts())
print("Rows before:", len(enriched_df))
print("Rows after:", len(final_df))

In [ ]:
# Validate join

assert (final_df["_merge"] == "both").all()
assert len(final_df) == len(enriched_df)
assert final_df["Income deprivation rate"].notna().all()

assert not final_df.duplicated(subset=["Reported by", "Month", "Crime type"]).any()

final_df = final_df.drop(columns="_merge")
final_df.head()

## 7. Final Validation and Export

In [ ]:
# standardise reporting column names
final_df = final_df.rename(columns={
    "Reported by": "police_force",
    "Month": "month",
    "Crime type": "crime_category",
    "Incident count": "incident_count",
    "Year": "year",
    "Population": "population",
    "Incident type": "incident_type",
    "Rate per 1000": "rate_per_1000",
    "Income deprivation rate": "income_deprivation_rate"})

#recalculate rates to retain precision
final_df["rate_per_1000"] = (final_df["incident_count"] / final_df["population"] * 1000).round(4)

In [ ]:
# Validation Cchecks

# Check that required fields contain no missing values
assert final_df.notna().all().all()

# Ensure one row per reporting combination
assert not final_df.duplicated(subset=["police_force", "month", "crime_category"]).any()

# Confirm totals reconcile with cleaned records
assert final_df["incident_count"].sum() == len(cleaned_df)

# Check expected reporting coverage
assert final_df["police_force"].nunique() == 4
assert final_df["month"].nunique() == 24

# Confirm valid values
assert final_df["population"].gt(0).all()
assert final_df["incident_count"].ge(0).all()
assert final_df["income_deprivation_rate"].between(0, 1).all()

print("final validation checks passed.")
print("Final dataset shape:", final_df.shape)

In [ ]:
# Export final csv
final_df.to_csv("crime_reporting_dataset.csv", index=False)
print("Dataset exported successfully.")

### Note to self

Committing more frequently now because i just remembered git is a thing.

### To-do

- [ ] make ingestion process files in batches rather than loading everything into memory
- [ ] Replace hard coded numbers in validation checks where appropriate
- [ ] review and tidy notebook.
- [ ] Statement of Work
- [ ] Data dictionary for final reporting dataset.
- [ ] Workflow diagram, show: ingestion -> cleaning -> transformation -> enrichment -> export
- [Y] Add geographical hotspot data (LSOA data) to answer Q8 "Are there geographic clusters ect."  
- [ ] Run the entire thing from a fresh kernel to make sure it's reproducible. - yes since the LSOA wards were added !!
- [ ] Check final CSV and confirm all deliverables meet the marking criteria.
- [ ] Commit final project to GitHub.
- [ ] **DELETE THIS block before submission**

maybe not in that order 

In [ ]:
#Add geographical hotspot data?
# Checking if codes overlap

crime_lsoas = set(cleaned_df["LSOA code"].dropna().unique())

population_lsoas = set(deprivation_df["LSOA code (2021)"].dropna().unique())

matching = crime_lsoas & population_lsoas

print("Crime LSOAs:", len(crime_lsoas))
print("Population LSOAs:", len(population_lsoas))
print("Matching codes:", len(matching))
print("Unmatched crime codes:", len(crime_lsoas - population_lsoas))

In [ ]:
# Taking a closer look
unmatched_records = cleaned_df[cleaned_df["LSOA code"].isin(crime_lsoas - population_lsoas)]

print("Records in unmatched LSOAs:", len(unmatched_records))
print("Percentage of cleaned records:", round(len(unmatched_records) / len(cleaned_df) * 100, 2))
print(unmatched_records["Reported by"].value_counts())

# only 293/436,692 unmatched LSOA codes

In [ ]:
unmatched_records[["LSOA code", "LSOA name", "Reported by"]].drop_duplicates().head(20)
# interesting apparently Warwickshire police has somehow acquired Cardiff and Powys(where my parents live!)
# and Gloucestershire is policing Anglesey Pembrokeshire and Carmarthenshire

In [ ]:
# Let's see if they're all in Wales 
# Spoiler they are 
unmatched_records["LSOA code"].str[0].value_counts()

In [ ]:
#deprivation dataset and crime data set use same LSOA codes
[col for col in deprivation_df.columns if "LSOA" in col]

In [ ]:
# Check same code also has the same LSOA name in both datasets
name_check = (cleaned_df[["LSOA code", "LSOA name"]].drop_duplicates().merge(
    deprivation_df[["LSOA code (2021)", "LSOA name (2021)"]],
    left_on="LSOA code",
    right_on="LSOA code (2021)",
    how="inner",
    validate="many_to_one"))

name_check["names_match"] = (name_check["LSOA name"] == name_check["LSOA name (2021)"])

print(name_check["names_match"].value_counts())

#All 2534 matching English LSOA codes also have identical names

In [ ]:
# Check missiing codes

missing_location = cleaned_df["LSOA code"].isna().sum()

print("Records with missing LSOA:", missing_location)
print("Percentage:", round(missing_location / len(cleaned_df) * 100, 2))

# 4694/431,705 records with no loaction data
# Final report use Matched/Welsh/LSOA/Unknown location? to retain all records but allow filtering 

In [ ]:
# Examine crimes with multiple recorded LSOAs
location_conflicts = (crime_df[crime_df["Crime ID"].notna()].groupby(["Crime ID", "Month", "Reported by"])["LSOA code"].nunique())

print("Crimes with conflicting LSOAs:", (location_conflicts > 1).sum())

In [ ]:

conflicting_locations = location_conflicts[location_conflicts > 1].index

print("Conflicting crime combinations:", len(conflicting_locations))

In [ ]:
# flag crimes with conflicting locations
cleaned_df["location_conflict"] = (cleaned_df.set_index(["Crime ID", "Month", "Reported by"]).index.isin(conflicting_locations))

print(cleaned_df["location_conflict"].value_counts())

In [ ]:
import numpy as np

cleaned_df["location_status"] = np.select([cleaned_df["location_conflict"], cleaned_df["LSOA code"].isna(), cleaned_df["LSOA code"].str.startswith("W", na=False)],
                                          ["Ambiguous location","Unknown location","Welsh LSOA"],
                                          default="Matched English LSOA")

print(cleaned_df["location_status"].value_counts())

# Validate
assert cleaned_df["location_status"].notna().all()
assert (cleaned_df["location_status"].value_counts().sum() == len(cleaned_df))

print("\nClassified successfully")

In [ ]:
valid_lsoas = set(deprivation_df["LSOA code (2021)"])

invalid_matches = cleaned_df[(cleaned_df["location_status"] == "Matched English LSOA") & ~cleaned_df["LSOA code"].isin(valid_lsoas)]

assert len(invalid_matches) == 0

print("matched english LSOAs validated")

In [ ]:
# Keep valid LSOAs group other records by location status
geo_df = cleaned_df.copy()

geo_df["reporting_lsoa"] = np.where(geo_df["location_status"] == "Matched English LSOA", geo_df["LSOA code"], geo_df["location_status"])

geo_reporting = (geo_df.groupby(["Reported by", "Month", "Crime type", "reporting_lsoa", "location_status"],dropna=False).size().reset_index(name="incident_count"))

print("Geographical reporting rows:", len(geo_reporting))
print("Total incidents:", geo_reporting["incident_count"].sum())

assert geo_reporting["incident_count"].sum() == len(cleaned_df)

In [ ]:
lsoa_enrichment = deprivation_df[["LSOA code (2021)", "LSOA name (2021)", "Total population: mid 2022", "Income Score (rate)"]].copy()

lsoa_enrichment = lsoa_enrichment.rename(columns={
    "LSOA code (2021)": "reporting_lsoa",
    "LSOA name (2021)": "lsoa_name",
    "Total population: mid 2022": "lsoa_population_2022",
    "Income Score (rate)": "lsoa_income_deprivation_rate"})

assert lsoa_enrichment["reporting_lsoa"].is_unique
assert lsoa_enrichment["lsoa_population_2022"].gt(0).all()
assert lsoa_enrichment["lsoa_income_deprivation_rate"].between(0, 1).all()

In [ ]:
#Join
geo_enriched = geo_reporting.merge(
    lsoa_enrichment,
    on="reporting_lsoa",
    how="left",
    validate="many_to_one",
    indicator=True)

print(geo_enriched["_merge"].value_counts())

assert len(geo_enriched) == len(geo_reporting)

assert (geo_enriched.loc[geo_enriched["location_status"] == "Matched English LSOA", "_merge"] == "both").all()

In [ ]:
# local incident rates
geo_enriched["lsoa_rate_per_1000"] = (geo_enriched["incident_count"] / geo_enriched["lsoa_population_2022"]* 1000)

geo_enriched.loc[geo_enriched["location_status"] != "Matched English LSOA","lsoa_rate_per_1000"] = np.nan

print("geographical rows:", len(geo_enriched))
print("total incidents:", geo_enriched["incident_count"].sum())
print("Rows without local rates:", geo_enriched["lsoa_rate_per_1000"].isna().sum())

In [ ]:
# Aggregate geographical counts back to the original reporting grain
geo_totals = (geo_enriched.groupby(["Reported by", "Month", "Crime type"])["incident_count"].sum().sort_index())

original_totals = (reporting_df.groupby(["Reported by", "Month", "Crime type"])["Crime count"].sum().sort_index())

# compare every force/month/category combination
pd.testing.assert_series_equal(geo_totals, original_totals, check_names=False)

print("Geographical totals match OG reporting totals.")

In [ ]:
force_enrichment = final_df[["police_force", "year","population","income_deprivation_rate"]].drop_duplicates()

# 1 record per police force and year
assert not force_enrichment.duplicated(subset=["police_force", "year"]).any()

print(force_enrichment.shape)

In [ ]:
geo_enriched["year"] = geo_enriched["Month"].str[:4].astype(int)

force_enrichment = force_enrichment.rename(columns={"population": "force_population","income_deprivation_rate": "force_income_deprivation_rate"})

In [ ]:
geo_final = geo_enriched.merge(
    force_enrichment,
    left_on=["Reported by", "year"],
    right_on=["police_force", "year"],
    how="left",
    validate="many_to_one",
    indicator="force_merge")

print(geo_final["force_merge"].value_counts())
print("rows before:", len(geo_enriched))
print("rows after:", len(geo_final))

In [ ]:
#Validate
assert (geo_final["force_merge"] == "both").all()
assert len(geo_final) == len(geo_enriched)
assert geo_final["force_population"].gt(0).all()
assert geo_final["force_income_deprivation_rate"].between(0, 1).all()
assert geo_final["incident_count"].sum() == len(cleaned_df)

print("Validated.")

In [ ]:
geo_final = geo_final.rename(columns={"Month": "month", "Crime type": "crime_category", "lsoa_rate_per_1000": "lsoa_monthly_rate_per_1000"})

# Actual LSOA codes only 
geo_final["lsoa_code"] = geo_final["reporting_lsoa"].where(geo_final["location_status"] == "Matched English LSOA")

geo_final["incident_type"] = np.where(geo_final["crime_category"] == "Anti-social behaviour","ASB","Recorded crime")

reporting_columns = [
    "police_force",
    "year",
    "month",
    "crime_category",
    "incident_type",
    "location_status",
    "lsoa_code",
    "lsoa_name",
    "incident_count",
    "lsoa_population_2022",
    "lsoa_income_deprivation_rate",
    "lsoa_monthly_rate_per_1000",
    "force_population",
    "force_income_deprivation_rate"]

geo_final_csv = geo_final[reporting_columns].copy()
geo_final_csv.head()

# OKAY nice but i want to improve "035B" not very readable for a dash

In [ ]:
### Ward lookup exploration

ward_file = "LSOA_(2021)_to_Ward_to_Local_Authority_District_(May_2022)_Best_Fit_Lookup_for_EW.csv"
ward_df = pd.read_csv(ward_file)

print(ward_df.shape)
print(ward_df.columns.tolist())
ward_df.head()

# I need WD22NM and LSOA21NM

In [ ]:
ward_df[ward_df["LTLA22NM"] == "Bournemouth, Christchurch and Poole"][["LSOA21NM", "WD22NM"]].head(20)

In [ ]:
ward_df[ward_df["LSOA21NM"] == "Bournemouth, Christchurch and Poole 036A"][["LSOA21CD", "LSOA21NM", "WD22NM", "LTLA22NM"]]
# great so 036A is Boscombe East & Pokesdown, I now know where that is but this is best fit not concerete - put that on the DASH 

In [ ]:
ward_lookup = ward_df[["LSOA21CD", "WD22NM"]].rename(columns={"LSOA21CD": "lsoa_code","WD22NM": "ward_name"}) # better names 
assert ward_lookup["lsoa_code"].is_unique

In [ ]:
# join to 
geo_final_csv = geo_final_csv.merge(
    ward_lookup,
    on="lsoa_code",
    how="left",
    validate="many_to_one",
    indicator=True)

print(geo_final_csv["_merge"].value_counts())

matched = geo_final_csv["location_status"] == "Matched English LSOA"

In [ ]:
# Validate ward join
matched = geo_final_csv["location_status"] == "Matched English LSOA"

assert len(geo_final_csv) == len(geo_final)
assert geo_final_csv["incident_count"].sum() == len(cleaned_df)
assert geo_final_csv.loc[matched, "ward_name"].notna().all()

#remove temp join indicator
geo_final_csv = geo_final_csv.drop(columns="_merge")

geo_final_csv[["lsoa_name", "ward_name", "incident_count"]].head(10)

In [ ]:
geo_final_csv[["local_authority", "lsoa_identifier"]] = (geo_final_csv["lsoa_name"].str.extract(r"^(.*) (\d{3}[A-Z])$")) #plit authority name and lsoa id 
geo_final_csv.head(10)